<a id="misc-recursion"></a>
### Misc — recursion patterns (important only)

Same rules: **base case** → **self-call on smaller input** → **return/combine**.
Binary / merge / quick already cover range-splitting. These are the short extras papers invent.


In [2]:
def Fact(n):
    # base: 0! / 1! = 1
    if n <= 1:
        return 1
    return n * Fact(n - 1)

def Sum1toN(n):
    # base: sum to 0 is 0
    if n <= 0:
        return 0
    return n + Sum1toN(n - 1)

def ReverseString(s):
    # base: empty or single char
    if len(s) <= 1:
        return s
    # last char + reverse of the rest
    return s[-1] + ReverseString(s[:-1])



# Search 'n Sort — S1B & S1C

Revision / content-lookup notebook for the searching and sorting algorithms in H2 Computing (9569).

## Contents
1. [Searching](#searching): linear, binary (recursive + iterative) and hash-table search
2. [Sorting](#sorting): insertion, bubble, quicksort and mergesort
3. [Misc recursion](#misc-recursion)
4. [Complexity lookup](#complexity-lookup)
5. [Test scaffold](#tests)

**How to use:** keep the logic and stopping conditions in memory; use the code as an exam-style implementation reference. Run the test scaffold after changing an algorithm.


In [3]:
# random test list of length n
import random
def reset_test_list():
    global test_list
    test_list = []
    n = 10
    lower_bound = 0
    upper_bound = n
    for i in range(n):
        test_list.append(random.randint(lower_bound, upper_bound))
    print(f"test list: {test_list}")
reset_test_list()



test list: [8, 10, 3, 4, 6, 1, 0, 5, 2, 9]


<a id="searching"></a>
## Searching

### Linear search

Probe everything in order along the line, 1 by 1 (hence linear). It works on **sorted or unsorted** data and returns the first matching index in this implementation.

**Stopping conditions:** return the index when found; return `-1` after the entire list has been searched.

- Best time: **O(1)** — found on the first term
- Average/worst time: **O(n)** — may sweep most/all of the list
- Extra space: **O(1)**

In [4]:
def LinearSearch(Arr, FindValue):
    for i in range(len(Arr)):
        if Arr[i] == FindValue:
            return i  # first match index
    return -1  # not found



### Binary search

Only works on a **sorted** list, but is way faster than linear search for large inputs. Take `FindValue`, compare it to the middle, then move left or right. This halves the search scope every pass until the target is found or the bounds cross.

Same logic in two shapes — learn **both**; papers may ask either.

#### Recursive (`BinarySearch`)

**Stopping conditions:**
- `Low > High` → absent, return `-1`
- middle matches → return its index
- otherwise recurse with `middle - 1` or `middle + 1` (bounds must shrink)

Extra space: **O(log n)** call-stack frames.

#### Iterative (`BinarySearchIterative`)

Same comparisons, but a `while Low <= High` loop updates `Low` / `High` in place. No recursive calls → extra space **O(1)**. Prefer this when depth/stack is a concern; prefer recursive when the question wants a recursion/stack trace.

- Best time: **O(1)** — middle matches immediately
- Average/worst time: **O(log n)**
- Needs sorted input: **yes**

In [5]:
# recursive binary search
def BinarySearch(Arr, FindValue, Low, High):
    # empty window - absent
    if Low > High:
        return -1
    middle = (Low + High) // 2  # mid index of current window
    if Arr[middle] == FindValue:
        return middle  # found
    elif FindValue < Arr[middle]:
        # go left: discard middle and right
        return BinarySearch(Arr, FindValue, Low, middle - 1)
    else:
        # go right: discard middle and left
        return BinarySearch(Arr, FindValue, middle + 1, High)
# +-1 drops middle - already checked, not in the remaining region

# iterative binary search - same decisions, loop instead of call stack
def BinarySearchIterative(Arr, FindValue, Low, High):
    while Low <= High:  # until window empty
        middle = (Low + High) // 2
        if Arr[middle] == FindValue:
            return middle
        elif FindValue < Arr[middle]:
            High = middle - 1  # move left
        else:
            Low = middle + 1  # move right
    return -1  # bounds crossed - not in list



### Hash-table search

Create a hash table and use a hash function to calculate the bucket/index to inspect. Collisions must be handled; the example below uses **separate chaining** (a list at each bucket).

- Average search time: **O(1)** when keys are spread well
- Worst search time: **O(n)** when many keys collide into one chain
- Extra space for one search: **O(1)**
- Storage for the whole hash table: **O(n)**

This is not the main exam-syntax version; for more comprehensive hash-table code, look at the ADT notebook. The important distinction is that O(1) is the **average**, not a guarantee.

In [6]:
# hashing with separate chaining (LLM-written early 2025; prefer ADT.ipynb HashTable)
BUCKET_SIZE = 7
class Hash(object):
    def __init__(self, bucket):
        self.__bucket = bucket  # number of buckets
        self.__table = [[] for _ in range(bucket)]  # one list per bucket

    def hashFunction(self, key):
        return (key % self.__bucket)

    def insertItem(self, key):
        index = self.hashFunction(key)
        self.__table[index].append(key)

    def deleteItem(self, key):
        index = self.hashFunction(key)
        if key not in self.__table[index]:
            return
        self.__table[index].remove(key)

    def displayHash(self):
        for i in range(self.__bucket):
            print("[%d]" % i, end='')
            for x in self.__table[i]:
                print(" --> %d" % x, end='')
            print()


# drive
if __name__ == "__main__":
    reset_test_list()
    h = Hash(BUCKET_SIZE)
    for x in test_list:
        h.insertItem(x)
    h.deleteItem(x)  # deletes last inserted x from the loop
    h.displayHash()



test list: [2, 7, 10, 6, 3, 2, 1, 3, 0, 2]
[0] --> 7 --> 0
[1] --> 1
[2] --> 2 --> 2
[3] --> 10 --> 3 --> 3
[4]
[5]
[6] --> 6


<a id="sorting"></a>
## Sorting

### Insertion sort

Front of list is the sorted portion and back is unsorted. Take the next object from the unsorted portion and compare it backwards through the sorted portion. Shift larger objects right until its insertion point is reached. This sweeps the list until the sorted portion is the size of the whole list.

**Useful invariant:** before each outer-loop pass, everything left of `i` is already sorted.

- Best time: **O(n)** — already sorted, so no shifting
- Average/worst time: **O(n²)**
- Extra space: **O(1)** (in-place)
- **Stable:** yes, because equal values are not shifted past each other

In [7]:
def InsertionSort(Arr):
    for i in range(1, len(Arr)):
        current = Arr[i]  # next item from unsorted portion
        j = i - 1
        # shift larger sorted items right until insertion spot found
        while j >= 0 and Arr[j] > current:
            Arr[j + 1] = Arr[j]  # shift right; leaves a hole behind
            j -= 1  # walk left
        # j stepped one past the spot, so hole is at j+1
        Arr[j + 1] = current  # drop into sorted portion
    return Arr



### Bubble sort

On every pass, look at the current object and next object. Swap to get them in the correct order, then move to the next index. Larger objects "bubble" towards the end. Keep passing until a full pass returns 0 swaps, which is when you know the whole list is sorted.

- Best time: **O(n)** with the `swapped` early-exit flag
- Average/worst time: **O(n²)**
- Extra space: **O(1)** (in-place)
- **Stable:** yes, when swapping only if left `>` right

Exam optimisation: after each pass, one more final element is fixed, so the inner range may be shortened. The current code is still correct without that optimisation.

In [8]:
def BubbleSort(Arr):
    n = len(Arr)
    swapped = True
    while swapped:
        swapped = False
        for i in range(n - 1):  # one pass over adjacent pairs
            if Arr[i] > Arr[i + 1]:  # pair out of order - swap
                temp = Arr[i]
                Arr[i] = Arr[i + 1]
                Arr[i + 1] = temp
                swapped = True  # something moved; need another pass
        # full pass with no swaps -> already sorted
    return Arr



### Quicksort

A divide-and-conquer algorithm: recursively partition parts of the list to conquer each part bit by bit.

**Partition:** take the first value as `pivotValue`. Move `leftmark` until it finds a value larger than the pivot; move `rightmark` until it finds a value smaller than the pivot. Swap those misplaced objects and continue until the bounds cross. Finally swap the pivot into `rightmark`. The pivot is now in its **final sorted position**: values on its left are ≤ it and values on its right are ≥ it.

**Conquer:** recursively quicksort the left and right sub-arrays. Once a sub-array has zero or one object (`First >= Last`), it is already sorted and recursion stops. Every partition fixes another pivot, eventually sorting the entire list.

Quicksort is recursive, with this implementation processing left then right to sweep through everything.

- Best/average time: **O(n log n)** with reasonably balanced partitions
- Worst time: **O(n²)** with very unbalanced partitions (e.g. consistently poor pivot choices)
- Array storage: in-place; recursion stack **O(log n)** average, **O(n)** worst
- Usually **not stable**

In [ ]:
def partition(arr, first, last):
    pivotvalue = arr[first]  # first value is pivot
    leftmark = first + 1
    rightmark = last
    done = False
    while not done:  # until marks cross (rightmark < leftmark)
        # bounds check before arr[...] — avoids IndexError
        while leftmark <= rightmark and arr[leftmark] <= pivotvalue:
            # move right until > pivot or marks meet
            leftmark += 1
        while rightmark >= leftmark and arr[rightmark] >= pivotvalue:
            # move left until < pivot or marks meet
            rightmark -= 1
        if rightmark < leftmark:
            done = True  # crossed - stop swapping
        else:
            # swap the two misplaced values
            temp = arr[leftmark]
            arr[leftmark] = arr[rightmark]
            arr[rightmark] = temp
    # park pivot at rightmark (final sorted index)
    arr[first] = arr[rightmark]
    arr[rightmark] = pivotvalue
    return rightmark  # split for the two halves

def quickSort(array, first, last):
    if first < last:
        split = partition(array, first, last)  # pivot fixed at split
        quickSort(array, first, split - 1)  # left half
        quickSort(array, split + 1, last)  # right half
    return array



### Mergesort

Divide and conquer: split into a binary tree, then sort by merging.

**Divide:** take the list and split it in half until you get single-object arrays (which are considered sorted). From there, your sorted arrays are ready to conquer.

**Conquer:** merge two selected arrays by comparing their current values. Copy the smaller number into the main array and move that side's index forward; when one side runs out, copy the leftovers from the other. Tada — you have a merged (and sorted) array.

Since mergesort is recursive (usually), merging the binary tree happens as you exit the recursion calls, and you end up with 1 big merged and sorted array.

- Best/average/worst time: **O(n log n)**
- Extra space: **O(n)** for temporary arrays; not in-place in this implementation
- **Stable:** yes here, because equal values are taken from `left` first (`<=`)

### Mergesort

Divide and conquer: split into a binary tree, then sort by merging.

**Divide:** take the list and split it in half until you get single-object arrays (which are considered sorted). From there, your sorted arrays are ready to conquer.

**Conquer:** merge two selected arrays by comparing their current values. Copy the smaller number into the main array and move that side's index forward; when one side runs out, copy the leftovers from the other. Tada — you have a merged (and sorted) array.

Since mergesort is recursive (usually), merging the binary tree happens as you exit the recursion calls, and you end up with 1 big merged and sorted array.

- Best/average/worst time: **O(n log n)**
- Extra space: **O(n)** for temporary arrays; not in-place in this implementation
- **Stable:** yes here, because equal values are taken from `left` first (`<=`)

In [10]:
def merge(array, low, mid, high):
    # copy the two halves (same array still holds both sections)
    left = array[low : mid + 1]
    right = array[mid + 1 : high + 1]
    # left/right are already sorted runs from earlier merges
    # copies mean this merge is NOT in-place
    l = 0  # front of left
    r = 0  # front of right
    a = low  # write-back index into array
    # take smaller head until one side empties
    while l < len(left) and r < len(right):
        if left[l] <= right[r]:
            array[a] = left[l]
            l += 1
        else:
            array[a] = right[r]
            r += 1
        a += 1
    # drain leftovers
    while l < len(left):
        array[a] = left[l]
        l += 1
        a += 1
    while r < len(right):
        array[a] = right[r]
        r += 1
        a += 1
    return array  # merged sorted range [low..high]

def mergeSort(array, low, high):
    if low < high:  # single element already sorted
        # divide
        mid = (low + high) // 2  # same middle as binary search
        mergeSort(array, low, mid)  # left half
        mergeSort(array, mid + 1, high)  # right half
        # conquer - merge on the way back up the call stack
        merge(array, low, mid, high)
    return array  # always the same array object, sorted in its range



<a id="complexity-lookup"></a>
## Complexity lookup

| Algorithm | Best | Average | Worst | Extra space | Needs sorted input? |
|---|---:|---:|---:|---:|---|
| Linear search | O(1) | O(n) | O(n) | O(1) | No |
| Binary search (recursive) | O(1) | O(log n) | O(log n) | O(log n) stack | Yes |
| Binary search (iterative) | O(1) | O(log n) | O(log n) | O(1) | Yes |
| Hash-table search | O(1) avg. | O(1) avg. | O(n) | O(1) per search | No |
| Insertion sort | O(n) | O(n²) | O(n²) | O(1) | — |
| Bubble sort (early exit) | O(n) | O(n²) | O(n²) | O(1) | — |
| Quicksort | O(n log n) | O(n log n) | O(n²) | O(log n) avg. stack | — |
| Mergesort | O(n log n) | O(n log n) | O(n log n) | O(n) | — |

### Exam distinctions
- **Search result:** return an index when found and a sentinel such as `-1` when absent.
- **In-place:** mutates the given list using O(1) extra array storage.
- **Stable:** equal-key records keep their original relative order.
- **Divide and conquer:** divide into smaller subproblems, solve recursively, combine.

<a id="tests"></a>
## Test scaffold
Run all definition cells above first. Copies (`test_list[:]`) stop one in-place sort affecting the next test.

In [23]:
# scaffold — run definition cells above first
reset_test_list()
print("linear:", LinearSearch(test_list, test_list[0]))

sorted_for_binary = sorted(test_list[:])  # binary needs sorted input
print("binary on", sorted_for_binary, "->", BinarySearch(sorted_for_binary, test_list[0], 0, len(sorted_for_binary) - 1))
print("binary iterative ->", BinarySearchIterative(sorted_for_binary, test_list[0], 0, len(sorted_for_binary) - 1))
print("binary miss ->", BinarySearch(sorted_for_binary, -1, 0, len(sorted_for_binary) - 1),
      BinarySearchIterative(sorted_for_binary, -1, 0, len(sorted_for_binary) - 1))

a = test_list[:]
print("insertion:", InsertionSort(a))

b = test_list[:]
print("bubble:", BubbleSort(b))

c = test_list[:]
print("quick:", quickSort(c, 0, len(c) - 1))

d = test_list[:]
print("merge:", mergeSort(d, 0, len(d) - 1))



test list: [8, 5, 4, 9, 4, 9, 1, 1, 7, 0]
linear: 0
binary on [0, 1, 1, 4, 4, 5, 7, 8, 9, 9] -> 7
binary iterative -> 7
binary miss -> -1 -1
insertion: [0, 1, 1, 4, 4, 5, 7, 8, 9, 9]
bubble: [0, 1, 1, 4, 4, 5, 7, 8, 9, 9]
quick: [0, 1, 1, 4, 4, 5, 7, 8, 9, 9]
merge: [0, 1, 1, 4, 4, 5, 7, 8, 9, 9]
